# LightGCN — bipartite user–plan graph recommender

Trains a LightGCN model on Agora's bipartite user–plan interaction graph and
exports the result back to Postgres. Per `AGENTS.md`: training is always
offline (here), the backend only ever does inference — read stored vectors,
dot product.

**Design recap** (see `AGENTS.md` and the Tier 1 recommender in
`agora/backend/application/recommendation.py` for the system this sits
alongside):
- Graph embeddings are **initialized** from each plan/user's existing
  semantic (Gemini) embedding, projected down to the graph's dimension —
  not random noise. This matters because the graph is small and sparse; a
  plan touched by one or two users barely has enough co-consumption evidence
  to learn a meaningful embedding from structure alone.
- A regularization term keeps every embedding **anchored** to that semantic
  init throughout training — real co-consumption evidence is free to pull an
  embedding away from it, but a node with almost no evidence just stays near
  its semantic starting point instead of drifting on noise.
- At serving time the backend blends the graph score with the semantic
  score, and cold-start plans (zero interactions, never a node in this
  graph) get a proxy embedding: the similarity-weighted average of their
  nearest neighbors' *graph* embeddings in semantic space.

**Caveat carried over from `scripts/generate_synthetic_interactions.py`:**
most of this graph's structure currently comes from hand-picked synthetic
archetypes, not real multi-user behavior. This notebook's job right now is
to prove the *pipeline* is sound — trains, evaluates, exports cleanly — not
to prove the graph model beats Tier 1 on real taste. Re-run once real
interaction volume exists and let the held-out numbers make that call.

In [1]:
# Jupyter sets the kernel's cwd to this notebook's own directory
# (notebooks/), not the repo root — but Settings() reads `.env` relative to
# cwd at import time (agora/backend/infrastructure/config.py), so without
# this, database_url silently ends up empty and every DB call hangs until
# it times out. Idempotent, so re-running this cell is safe.
import os
if not os.path.exists("AGENTS.md"):
    os.chdir("..")
assert os.path.exists("AGENTS.md"), "expected to land at the project root"

In [2]:
import json
import random

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from agora.backend.infrastructure.persistence.postgres_repository import (
    get_all_city_plans,
    get_all_interactions_for_city,
    init_db,
    pool,
    set_plan_graph_embeddings_bulk,
    upsert_user_embeddings_bulk,
)

CITY = "Madrid"
SEED = 0

DIM = 64            # embedding dimension — small on purpose; ~3K edges can't support much more
K_LAYERS = 3         # LightGCN propagation depth
LAMBDA_REG = 1e-3    # semantic-anchor regularization strength
LR = 0.01
EPOCHS = 150
COLDSTART_K = 5      # neighbors averaged for plans with zero interactions
ALPHA = 0.5          # graph vs. semantic blend weight, evaluated below — tune once real data exists

INTERACTION_WEIGHT = {"saved": 3.0, "view_link": 2.0, "click": 1.0}  # mirrors ranking.py

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

## Load the bipartite graph from Postgres

Every deduped `(user_id, plan_id)` tie for the city — a user can have
`click`/`view_link`/`saved` rows on the same plan, but that's still one edge
for graph *structure* purposes (see `scripts/inspect_interaction_graph.py`,
which does the same dedup for its diagnostics). We keep the interaction
*types* around too, for weighting the semantic init below.

In [3]:
init_db()
plans = get_all_city_plans(CITY)
plan_by_id = {p["id"]: p for p in plans}

edges_raw = get_all_interactions_for_city(CITY)
ties_types: dict[tuple[str, int], set[str]] = {}
for e in edges_raw:
    if e["plan_id"] not in plan_by_id:
        continue  # stale/deleted plan still referenced by an old interaction row
    ties_types.setdefault((e["user_id"], e["plan_id"]), set()).add(e["interaction_type"])

ties = sorted(ties_types)
user_ids = sorted({u for u, _ in ties})
plan_ids = sorted({p for _, p in ties})
user_idx = {u: i for i, u in enumerate(user_ids)}
plan_idx = {p: i for i, p in enumerate(plan_ids)}
n_users, n_plans = len(user_ids), len(plan_ids)

print(f"users={n_users}  plans={n_plans}  ties={len(ties)}")

users=473  plans=973  ties=7531


## Leave-one-out train/val split

For every user with ≥3 interactions, hold out one random tie as validation
ground truth. This is the only thing that lets us later ask "did the model
learn anything" rather than just "did the loss go down."

In [4]:
rng = random.Random(SEED)
by_user: dict[str, list[int]] = {}
for u, p in ties:
    by_user.setdefault(u, []).append(plan_idx[p])

train_edges: list[tuple[int, int]] = []
val_edges: list[tuple[int, int]] = []  # (user_idx, held_out_plan_idx)
for u, plist in by_user.items():
    if len(plist) >= 3:
        held = rng.choice(plist)
        val_edges.append((user_idx[u], held))
        train_edges.extend((user_idx[u], p) for p in plist if p != held)
    else:
        train_edges.extend((user_idx[u], p) for p in plist)

train_by_user: dict[int, set[int]] = {}
for u, p in train_edges:
    train_by_user.setdefault(u, set()).add(p)

print(f"train_edges={len(train_edges)}  val_edges={len(val_edges)}")
assert all(u in train_by_user for u in range(n_users)), "a user has zero train edges"
for u, p in val_edges:
    assert p not in train_by_user[u], "val plan leaked into train"

train_edges=7073  val_edges=458


## Semantic init — project Gemini embeddings into graph-embedding space

Plans already carry a Gemini text embedding (`plans.embedding`, backfilled
by Tier 1's `backfill_embeddings`) — 3072-d. We fit a PCA down to `DIM` on
the touched plans' vectors and use that as every plan's *starting* graph
embedding. Users start from their weighted semantic profile (same recipe as
`user_profile()` in `ranking.py`), built only from **train** edges — the
held-out plan must never leak into a user's init.

In [5]:
gemini_dim = None
plan_gemini: dict[int, np.ndarray] = {}
for pid, i in plan_idx.items():
    emb = plan_by_id[pid].get("embedding")
    if emb:
        v = np.array(json.loads(emb), dtype=np.float64)
        plan_gemini[i] = v
        gemini_dim = gemini_dim or len(v)

print(f"plans with a Gemini embedding: {len(plan_gemini)}/{n_plans}  (dim={gemini_dim})")

user_gemini: dict[int, np.ndarray] = {}
for u_str, u in user_idx.items():
    num = np.zeros(gemini_dim)
    den = 0.0
    for p in train_by_user.get(u, []):
        if p not in plan_gemini:
            continue
        w = max(INTERACTION_WEIGHT.get(t, 1.0) for t in ties_types[(u_str, plan_ids[p])])
        num += plan_gemini[p] * w
        den += w
    if den > 0:
        user_gemini[u] = num / den

print(f"users with a semantic profile: {len(user_gemini)}/{n_users}")

plans with a Gemini embedding: 973/973  (dim=3072)
users with a semantic profile: 473/473


In [6]:
# PCA fit on the plans that actually have a Gemini vector; anything missing
# one (rare — a handful of very recent scrapes) falls back to small random
# init, same as vanilla LightGCN would do for every node.
plan_matrix = np.stack([plan_gemini[i] for i in sorted(plan_gemini)])
mean = plan_matrix.mean(axis=0)
_, _, vt = np.linalg.svd(plan_matrix - mean, full_matrices=False)
components = vt[:DIM]


def project(v: np.ndarray) -> np.ndarray:
    return (v - mean) @ components.T


plan_init = np.stack([
    project(plan_gemini[i]) if i in plan_gemini else np.random.normal(scale=0.1, size=DIM)
    for i in range(n_plans)
])
user_init = np.stack([
    project(user_gemini[i]) if i in user_gemini else np.random.normal(scale=0.1, size=DIM)
    for i in range(n_users)
])
print("plan_init", plan_init.shape, " user_init", user_init.shape)

plan_init (973, 64)  user_init (473, 64)


## Normalized bipartite adjacency

Symmetric, degree-normalized (`1/sqrt(deg_i · deg_j)`) — standard LightGCN,
no self-loops (the layer-combination step below supplies the "self"
contribution instead). Built from **train edges only**.

In [7]:
N = n_users + n_plans


def plan_node(p: int) -> int:
    return n_users + p


degree = np.zeros(N)
for u, p in train_edges:
    degree[u] += 1
    degree[plan_node(p)] += 1

rows, cols, vals = [], [], []
for u, p in train_edges:
    pn = plan_node(p)
    w = 1.0 / np.sqrt(degree[u] * degree[pn])
    rows += [u, pn]
    cols += [pn, u]
    vals += [w, w]

adj = torch.sparse_coo_tensor(
    torch.tensor([rows, cols], dtype=torch.long),
    torch.tensor(vals, dtype=torch.float32),
    size=(N, N),
    check_invariants=False,
).coalesce()
print("adjacency nnz:", adj._nnz(), " shape:", tuple(adj.shape))

adjacency nnz: 14146  shape: (1446, 1446)


## LightGCN

"Light" = no nonlinearities, no per-layer weight matrices, no self-loop —
just symmetric neighbor averaging, repeated `K_LAYERS` times, then a mean
across layer 0 (the node's own trainable embedding) through layer K (the
most diffuse hop of collaborative signal).

In [8]:
class LightGCN(nn.Module):
    def __init__(self, n_users: int, n_plans: int, dim: int, adj: torch.Tensor, k_layers: int):
        super().__init__()
        self.n_users = n_users
        self.adj = adj
        self.k_layers = k_layers
        self.user_emb = nn.Embedding(n_users, dim)
        self.plan_emb = nn.Embedding(n_plans, dim)

    def forward(self):
        x0 = torch.cat([self.user_emb.weight, self.plan_emb.weight], dim=0)
        layers = [x0]
        x = x0
        for _ in range(self.k_layers):
            x = torch.sparse.mm(self.adj, x)
            layers.append(x)
        out = torch.stack(layers, dim=0).mean(dim=0)
        return out[: self.n_users], out[self.n_users :]


model = LightGCN(n_users, n_plans, DIM, adj, K_LAYERS)
with torch.no_grad():
    model.user_emb.weight.copy_(torch.tensor(user_init, dtype=torch.float32))
    model.plan_emb.weight.copy_(torch.tensor(plan_init, dtype=torch.float32))

# frozen copies of the init, for the regularization term below
user_init_t = model.user_emb.weight.detach().clone()
plan_init_t = model.plan_emb.weight.detach().clone()

u_final, p_final = model()
print("forward OK:", tuple(u_final.shape), tuple(p_final.shape))

forward OK: (473, 64) (973, 64)


## BPR training loop

Bayesian Personalized Ranking: for every training edge `(u, p+)`, sample a
random plan `p-` the user hasn't touched, and push
`score(u, p+) > score(u, p-)`. Plus the semantic-anchor regularization term
described above. Full-graph propagation runs once per epoch (standard for a
transductive GCN); at this graph size (~800 nodes) that's milliseconds, no
GPU needed.

In [9]:
train_u = torch.tensor([u for u, _ in train_edges], dtype=torch.long)
train_p = torch.tensor([p for _, p in train_edges], dtype=torch.long)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)


def sample_negatives(gen: torch.Generator) -> torch.Tensor:
    neg = torch.randint(0, n_plans, (len(train_edges),), generator=gen)
    for i in range(len(neg)):
        u = train_edges[i][0]
        tries = 0
        while neg[i].item() in train_by_user[u] and tries < 10:
            neg[i] = torch.randint(0, n_plans, (1,), generator=gen).item()
            tries += 1
    return neg


g = torch.Generator().manual_seed(SEED)
for epoch in range(1, EPOCHS + 1):
    model.train()
    optimizer.zero_grad()
    u_emb, p_emb = model()
    neg_p = sample_negatives(g)

    pos_score = (u_emb[train_u] * p_emb[train_p]).sum(-1)
    neg_score = (u_emb[train_u] * p_emb[neg_p]).sum(-1)
    bpr_loss = -F.logsigmoid(pos_score - neg_score).mean()

    reg_loss = LAMBDA_REG * (
        (model.user_emb.weight - user_init_t).pow(2).sum(-1).mean()
        + (model.plan_emb.weight - plan_init_t).pow(2).sum(-1).mean()
    )
    loss = bpr_loss + reg_loss
    loss.backward()
    optimizer.step()

    if epoch % 25 == 0 or epoch == 1:
        print(f"epoch {epoch:4d}  bpr={bpr_loss.item():.4f}  reg={reg_loss.item():.4f}  total={loss.item():.4f}")

epoch    1  bpr=0.6872  reg=0.0000  total=0.6872


epoch   25  bpr=0.4169  reg=0.0058  total=0.4228


epoch   50  bpr=0.1751  reg=0.0210  total=0.1961


epoch   75  bpr=0.0926  reg=0.0332  total=0.1258


epoch  100  bpr=0.0609  reg=0.0407  total=0.1016


epoch  125  bpr=0.0450  reg=0.0451  total=0.0901


epoch  150  bpr=0.0346  reg=0.0478  total=0.0824


## Evaluation: Recall@10 vs. Tier 1 baseline

For each held-out `(user, plan)`: rank every plan that user hasn't trained
on, check whether the true held-out plan lands in the top 10. Compared three
ways — pure Tier 1 semantic (today's production baseline), pure graph, and
the blend — using full-dimension Gemini vectors for the semantic score so
it's a fair comparison against what Tier 1 actually serves today, not the
PCA-reduced init used only for graph training.

In [10]:
model.eval()
with torch.no_grad():
    u_final, p_final = model()
u_graph = F.normalize(u_final, dim=-1).numpy()
p_graph = F.normalize(p_final, dim=-1).numpy()

user_sem = np.zeros((n_users, gemini_dim))
has_user_sem = np.zeros(n_users, dtype=bool)
for i, v in user_gemini.items():
    user_sem[i] = v / (np.linalg.norm(v) + 1e-9)
    has_user_sem[i] = True

plan_sem = np.zeros((n_plans, gemini_dim))
has_plan_sem = np.zeros(n_plans, dtype=bool)
for i, v in plan_gemini.items():
    plan_sem[i] = v / (np.linalg.norm(v) + 1e-9)
    has_plan_sem[i] = True

K = 10


def semantic_score(u, candidates):
    if not has_user_sem[u]:
        return [-1.0] * len(candidates)
    return [float(user_sem[u] @ plan_sem[p]) if has_plan_sem[p] else -1.0 for p in candidates]


def graph_score(u, candidates):
    return [float(u_graph[u] @ p_graph[p]) for p in candidates]


def blended_score(u, candidates):
    g, s = graph_score(u, candidates), semantic_score(u, candidates)
    return [ALPHA * gi + (1 - ALPHA) * si for gi, si in zip(g, s)]


def recall_at_k(score_fn, name: str) -> float:
    hits = 0
    for u, held in val_edges:
        candidates = [p for p in range(n_plans) if p not in train_by_user[u]]
        scores = score_fn(u, candidates)
        ranked = [p for _, p in sorted(zip(scores, candidates), reverse=True)]
        hits += held in ranked[:K]
    recall = hits / len(val_edges)
    print(f"{name:>12s}  recall@{K} = {recall:.3f}  ({hits}/{len(val_edges)})")
    return recall


_ = recall_at_k(semantic_score, "semantic")
_ = recall_at_k(graph_score, "graph")
_ = recall_at_k(blended_score, "blended")

    semantic  recall@10 = 0.144  (66/458)


       graph  recall@10 = 0.181  (83/458)


     blended  recall@10 = 0.207  (95/458)


## Export: write trained + cold-start embeddings back to Postgres

Final embeddings are stored **raw**, not pre-normalized — same convention
`plans.embedding` already follows (`ranking.py`'s `cosine()` normalizes at
query time), so both embedding columns behave the same way for callers.

Plans with zero interactions never became graph nodes at all — they get a
proxy: a weighted average of a handful of their *trained* neighbors' graph
embeddings, found via cosine similarity in semantic (Gemini) space. Two
stages, not similarity alone:

1. **Similarity floor** — take the top `COLDSTART_K * SHORTLIST_MULT`
   candidates by raw similarity only. This is a genuine relevance
   requirement: nothing gets in that isn't actually alike.
2. **Confidence breaks ties within that shortlist** — `d / (d + CONFIDENCE_PRIOR)`,
   based on each neighbor's own training degree, picks the final `COLDSTART_K`
   and their weights from *within* the similarity-qualified pool.

Tried a flat `similarity * confidence` score first (still in git history) —
confidence alone could override a large similarity gap: a neighbor at only
0.55 similarity but 31 interactions beat one at 0.76 similarity with just 3,
dragging a cold-start plan toward a generic well-connected hub instead of
anything thematically relevant. The two-stage version fixes that: confidence
can only choose among candidates that already passed a real relevance bar,
never promote something irrelevant just because it's well-trained.

A plan with neither interactions nor a Gemini embedding gets no
`graph_embedding` at all; the backend falls back to semantic-only for it,
same shape as today's existing fallback for a plan with no `embedding`.

In [11]:
u_final_np = u_final.detach().numpy()
p_final_np = p_final.detach().numpy()

plan_export: dict[int, list[float]] = {plan_ids[i]: p_final_np[i].tolist() for i in range(n_plans)}
user_export: list[tuple[str, list[float]]] = [(user_ids[i], u_final_np[i].tolist()) for i in range(n_users)]

all_plans = get_all_city_plans(CITY)
touched_ids = set(plan_ids)
untouched = [p for p in all_plans if p["id"] not in touched_ids and p.get("embedding")]

# Similarity FLOOR, then confidence breaks ties within it — tried a flat
# similarity*confidence score first (still in git history), but confidence
# alone could override a huge similarity gap: for one plan, a neighbor at
# only 0.55 similarity but very high confidence (31 interactions) beat one
# at 0.76 similarity with 3 interactions, dragging the proxy toward a
# generic well-connected hub instead of anything thematically relevant.
# Two stages instead: (1) take the top COLDSTART_K * SHORTLIST_MULT
# candidates by RAW similarity only — a genuine relevance floor — then (2)
# within THAT already-similar shortlist, prefer the more confidently-
# trained ones (d / (d + CONFIDENCE_PRIOR), same shrinkage as before).
# Confidence can no longer promote something irrelevant; it can only choose
# among options that already look genuinely alike.
CONFIDENCE_PRIOR = 5   # interactions needed to reach 50% trust
SHORTLIST_MULT = 4     # similarity floor: only the top COLDSTART_K * this many candidates are eligible at all

if untouched:
    touched_ids_with_emb = [pid for pid in plan_ids if pid in plan_gemini]
    touched_matrix = np.stack([plan_gemini[plan_idx[pid]] for pid in touched_ids_with_emb])
    touched_norm = touched_matrix / (np.linalg.norm(touched_matrix, axis=1, keepdims=True) + 1e-9)
    touched_degree = np.array([degree[plan_node(plan_idx[pid])] for pid in touched_ids_with_emb])
    touched_confidence = touched_degree / (touched_degree + CONFIDENCE_PRIOR)
    shortlist_size = min(len(touched_ids_with_emb), COLDSTART_K * SHORTLIST_MULT)

    for plan in untouched:
        v = np.array(json.loads(plan["embedding"]), dtype=np.float64)
        v = v / (np.linalg.norm(v) + 1e-9)
        sims = touched_norm @ v
        shortlist = np.argsort(-sims)[:shortlist_size]
        shortlist_score = np.clip(sims[shortlist], 1e-6, None) * touched_confidence[shortlist]
        top_k_local = np.argsort(-shortlist_score)[:COLDSTART_K]
        top_k = shortlist[top_k_local]
        weights = shortlist_score[top_k_local] / shortlist_score[top_k_local].sum()
        neighbor_embs = np.stack([plan_export[touched_ids_with_emb[i]] for i in top_k])
        plan_export[plan["id"]] = (weights[:, None] * neighbor_embs).sum(axis=0).tolist()

print(
    f"exporting graph_embedding for {len(plan_export)}/{len(all_plans)} plans "
    f"({len(plan_ids)} trained + {len(plan_export) - len(plan_ids)} cold-start proxies)"
)
print(f"exporting user_embeddings for {len(user_export)} users")

set_plan_graph_embeddings_bulk(list(plan_export.items()))
upsert_user_embeddings_bulk(CITY, user_export)
pool.close()
print("export done")

exporting graph_embedding for 1003/1003 plans (973 trained + 30 cold-start proxies)
exporting user_embeddings for 473 users


export done
